# H5. Training and its failures
Book: Gradient Descent, sections on SGD and automatic differentiation.

Supplement: Alisa's book of LLMs, [Backpropagation](https://alisawuffles.notion.site/alisa-s-book-of-llms#2de7eb873605808084f0f9e73e97a2ac) and [Learning rate](https://alisawuffles.notion.site/alisa-s-book-of-llms#2e67eb87360580239773dcff11b7d150). Trace our small network before reading larger examples.
These selected readings are optional support. The classroom examples define the required scope.
Computing connection: [Alisa's Train memory use](https://alisawuffles.notion.site/alisa-s-book-of-llms#3027eb87360580ffa857e851e773b3f6).
Read the storage categories. Exact bytes depend on dtype, optimizer, and implementation.

The supplied model uses four hidden ReLU units and BCEWithLogitsLoss.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def circle_data(seed=601, n=200):
    rng = np.random.default_rng(seed)
    n0 = n // 2
    radius = np.r_[rng.uniform(0, .4, n0), rng.uniform(.8, 1, n-n0)]
    angle = rng.uniform(0, 2*np.pi, n)
    x = np.c_[radius*np.cos(angle), radius*np.sin(angle)]
    y = np.r_[np.zeros(n0), np.ones(n-n0)].astype(int)
    return x, y

def circle_split():
    from sklearn.model_selection import train_test_split
    x, y = circle_data()
    xa, xt, ya, yt = train_test_split(x, y, test_size=.2, stratify=y,
                                     random_state=600)
    xr, xv, yr, yv = train_test_split(xa, ya, test_size=.25, stratify=ya,
                                     random_state=600)
    return xr, xv, xt, yr, yv, yt

def radial_features(x):
    return np.sum(x*x, axis=1, keepdims=True)

def logistic_model(radial=False):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler, FunctionTransformer
    from sklearn.linear_model import LogisticRegression
    steps = [FunctionTransformer(radial_features)] if radial else []
    return make_pipeline(*steps, StandardScaler(), LogisticRegression(C=1.0))

def train_network(xr, yr, xv, yv, lr=.5, epochs=600, width=4,
                  seed=1, update=True, batch_size=None):
    import torch
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    model = torch.nn.Sequential(torch.nn.Linear(2, width), torch.nn.ReLU(),
                                torch.nn.Linear(width, 1))
    x = torch.tensor(xr, dtype=torch.float32)
    y = torch.tensor(yr[:, None], dtype=torch.float32)
    vx = torch.tensor(xv, dtype=torch.float32)
    vy = torch.tensor(yv[:, None], dtype=torch.float32)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = {"train": [], "validation": []}
    batch_size = len(y) if batch_size is None else batch_size
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y))
        for start in range(0, len(y), batch_size):
            idx = order[start:start+batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            if update:
                optimizer.step()
        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(x), y).item())
            history["validation"].append(loss_fn(model(vx), vy).item())
        if not np.isfinite(history["train"][-1]):
            break
    return model, history

def network_probability(model, x):
    import torch
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(x, dtype=torch.float32))).numpy().ravel()

def dense_cost(batch, inputs, outputs, dtype="float32"):
    """Approximate dense-matmul FLOPs and resident element bytes, without allocation."""
    for value in (batch, inputs, outputs):
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
            raise ValueError("Dimensions must be positive integers.")
    data_type = np.dtype(dtype)
    if data_type not in (np.dtype("float32"), np.dtype("float64")):
        raise ValueError("Use float32 or float64 for this experiment.")
    batch, inputs, outputs = int(batch), int(inputs), int(outputs)
    size = data_type.itemsize
    return {"flops": 2*batch*inputs*outputs, "input_bytes": size*batch*inputs,
            "weight_bytes": size*inputs*outputs, "output_bytes": size*batch*outputs}

def profile_training_step(batch=64, inputs=2, width=4, repeats=20, seed=600):
    """Time four stages of a fresh CPU toy model, excluding initialization and I/O."""
    import torch
    from time import perf_counter
    dense_cost(batch, inputs, width)
    if isinstance(repeats, bool) or not isinstance(repeats, (int, np.integer)) or repeats < 1:
        raise ValueError("Repeats must be a positive integer.")
    rng = np.random.default_rng(seed)
    data = rng.normal(size=(batch, inputs)).astype("float32")
    targets = (data.sum(axis=1, keepdims=True) > 0).astype("float32")
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed)
            model = torch.nn.Sequential(
                torch.nn.Linear(inputs, width, device="cpu", dtype=torch.float32),
                torch.nn.ReLU(),
                torch.nn.Linear(width, 1, device="cpu", dtype=torch.float32))
            optimizer = torch.optim.SGD(model.parameters(), lr=.01)
            loss_fn = torch.nn.BCEWithLogitsLoss()
            samples = []
            for repetition in range(repeats+3):
                start = perf_counter()
                x = torch.from_numpy(data.copy())
                y = torch.from_numpy(targets.copy())
                prepared = perf_counter()
                optimizer.zero_grad(set_to_none=True)
                loss = loss_fn(model(x), y)
                forwarded = perf_counter()
                loss.backward()
                backwarded = perf_counter()
                optimizer.step()
                updated = perf_counter()
                if repetition >= 3:
                    samples.append([prepared-start, forwarded-prepared,
                                    backwarded-forwarded, updated-backwarded])
            milliseconds = 1000*np.asarray(samples)
            names = ["prepare", "forward_loss_clear", "backward", "update"]
            parameter_bytes = sum(p.numel()*p.element_size() for p in model.parameters())
            gradient_bytes = sum(p.grad.numel()*p.grad.element_size() for p in model.parameters())
            return {"stages_ms": dict(zip(names, map(float, np.median(milliseconds, axis=0)))),
                    "step_median_ms": float(np.median(milliseconds.sum(axis=1))),
                    "parameter_bytes": parameter_bytes, "gradient_bytes": gradient_bytes}
    finally:
        torch.set_num_threads(previous_threads)

## The operations in one update
Read the loop before running it. Point to forward computation, the loss,
clearing gradients, backward computation, and the parameter update.

In [ ]:
import torch
torch.set_num_threads(1)
torch.manual_seed(1)
xr, xv, xt, yr, yv, yt = circle_split()
x = torch.tensor(xr, dtype=torch.float32)
y = torch.tensor(yr[:, None], dtype=torch.float32)
model = torch.nn.Sequential(torch.nn.Linear(2, 4), torch.nn.ReLU(),
                            torch.nn.Linear(4, 1))
optimizer = torch.optim.SGD(model.parameters(), lr=.5)
loss_fn = torch.nn.BCEWithLogitsLoss()
before = model[0].weight.detach().clone()
optimizer.zero_grad()
logits = model(x)
loss = loss_fn(logits, y)
loss.backward()
optimizer.step()
print("First-layer weight change:", model[0].weight.detach()-before)

## A1. One gradient, three ways (6 minutes)
For u=wx+b, h=ReLU(u), z=vh+c and binary cross-entropy, use:
x=2, y=1, w=.5, b=-.25, v=1.5, c=-.2.
Predict the gradient's sign and calculate the chain product before running.

In [ ]:
from scipy.special import expit
x_check, y_check, w_check, b_check, v_check, c_check = 2., 1., .5, -.25, 1.5, -.2
def scalar_loss(w):
    logit = v_check*max(0., w*x_check+b_check)+c_check
    return np.logaddexp(0., -logit)
u_check = w_check*x_check+b_check
z_check = v_check*max(0., u_check)+c_check
analytic = (expit(z_check)-y_check)*v_check*(u_check > 0)*x_check
eps = 1e-5
finite_difference = (scalar_loss(w_check+eps)-scalar_loss(w_check-eps))/(2*eps)
w_autograd = torch.tensor(w_check, dtype=torch.float64, requires_grad=True)
z_autograd = v_check*torch.relu(w_autograd*x_check+b_check)+c_check
loss_autograd = torch.nn.functional.binary_cross_entropy_with_logits(
    z_autograd, torch.tensor(y_check, dtype=torch.float64))
loss_autograd.backward()
automatic = w_autograd.grad.item()
print("Hand formula:", analytic, "finite differences:", finite_difference,
      "autograd:", automatic)
assert np.allclose([finite_difference, automatic], analytic, atol=1e-8)

Why did we keep u away from zero? Which method approximates a derivative?

Prediction:

Observation:

Explanation:

## A2. One learning rate at a time (24 minutes)
Predict the curve for each setting. All runs begin from the same seed.

In [ ]:
rates = [.01, .5, 10.]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), sharey=True)
for ax, rate in zip(axes, rates):
    fitted, history = train_network(xr, yr, xv, yv, lr=rate)
    print("Learning rate", rate, "final training loss", history["train"][-1],
          "final validation loss", history["validation"][-1])
    ax.plot(history["train"], label="Training")
    ax.plot(history["validation"], label="Validation")
    ax.set(title=f"Learning rate {rate}", xlabel="Epoch", ylim=(0, 1.5))
axes[0].set_ylabel("Mean cross-entropy")
axes[-1].legend()
plt.tight_layout()
plt.show()

A curve outside the display range is a reason to inspect its values.
Compare final losses, not only the plots. Does a lower training loss alone
prove that the model will generalize?

Prediction:

Observation:

Explanation:

## B1. The disabled update (20 minutes)
The model below calculates gradients but never changes its parameters.
Predict its loss curve. Repair it by setting update=True and explain the result.

In [ ]:
fitted, history = train_network(xr, yr, xv, yv, epochs=80, update=False)
plt.plot(history["train"], label="Training")
plt.plot(history["validation"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Mean cross-entropy")
plt.legend()
plt.show()

Explain why a successful backward pass did not produce learning.
Optional after class: make a controlled experiment with batch_size=16 in train_network.
Compare variability and note that epochs and update counts are different units.

Prediction:

Observation:

Explanation:

## B2. Where does a working training step spend time? (10 minutes)
Predict the slowest stage. The helper creates a fresh CPU network, preserving
the earlier model, random state, and final-test choices. Setup is outside timing.
Three warmup updates precede 20 measured updates. It uses one CPU thread.
Preparation means copying in-memory arrays and wrapping tensors, not file I/O.

In [ ]:
import platform
print("CPU platform:", platform.machine(), "PyTorch:", torch.__version__)
for hidden_width in [4, 128]:
    measurement = profile_training_step(width=hidden_width)
    print("\nHidden width:", hidden_width)
    for stage, milliseconds in measurement["stages_ms"].items():
        print(f"{stage:22s} {milliseconds:.5f} ms")
    print("Median whole step:", round(measurement["step_median_ms"], 5), "ms")
    print("Parameter element bytes:", measurement["parameter_bytes"],
          "gradient element bytes:", measurement["gradient_bytes"])

Inspect `profile_training_step` in Setup and identify each timing boundary.
Rerun and check whether the ranking is stable. Tiny operations include framework
and timing overhead. Per-stage medians need not sum to the median whole step.
No disk, network, GPU transfer, or production data-loading time is measured here.
The parameter/gradient byte counts exclude activations and other process memory.

Prediction:

Observation:

Candidate bottleneck and a confirming experiment:

### A limit on speedup
Hypothetical end-to-end time is 100 ms, including 30 ms in backward.
If backward becomes twice as fast, compute the new total and the overall speedup.
Why would a faster GPU fail to remove time spent waiting on a data source?
Reference: [PyTorch benchmarking guide](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html).
GPU timing requires device-aware synchronization or events; our experiment stays on CPU.

## Final test, after recording your choice
Set reveal_test=True only after locking the training settings. No further tuning.

In [ ]:
from sklearn.metrics import log_loss, accuracy_score
chosen_rate = .5
chosen_epochs = 600
reveal_test = False
if reveal_test:
    chosen, _ = train_network(xr, yr, xv, yv, lr=chosen_rate, epochs=chosen_epochs)
    raw = logistic_model().fit(xr, yr)
    radial = logistic_model(radial=True).fit(xr, yr)
    predictions = {"Raw logistic": raw.predict_proba(xt)[:, 1],
                   "Radial logistic": radial.predict_proba(xt)[:, 1],
                   "Network": network_probability(chosen, xt)}
    for name, probability in predictions.items():
        print(name, "log loss", log_loss(yt, probability),
              "accuracy", accuracy_score(yt, probability >= .5))

## Individual exit
Explain a flat training curve, unstable updates, and a widening training/validation gap.
For z=wx+b and y=1, how does the loss gradient change w when x is positive?
Which measured stage would you investigate first? What did the experiment exclude?